<a href="https://colab.research.google.com/github/felipefe/python-version/blob/stocks/stock_random_forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.tree import export_graphviz
import graphviz
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [12]:
# 1. Obter dados históricos semanais de longo prazo (ex: 8 anos para dar massa de dados)
ticker_ativo = "EGIE3.SA"
ticker_dolar = "USDBRL=X" #somente para acoes dolarizada (SUZANO, WEG, VALE)
ticker_dolar = "^BVSP"

print("Baixando dados históricos semanais...")
df_ativo = yf.download(ticker_ativo,start="2018-01-01",end="2026-11-10",interval="1wk",progress=False,)
df_dolar = yf.download(ticker_dolar,start="2018-01-01",end="2026-11-10",interval="1wk",progress=False,)

# Tratamento para multi-index no yfinance
if isinstance(df_ativo.columns, pd.MultiIndex):
  df_ativo.columns = df_ativo.columns.get_level_values(0)
if isinstance(df_dolar.columns, pd.MultiIndex):
  df_dolar.columns = df_dolar.columns.get_level_values(0)

# Unir o ativo e o Dólar em um único DataFrame alinhado pela data
df = pd.DataFrame()
df["Close"] = df_ativo["Close"]
df["Volume"] = df_ativo["Volume"]
df["Close_Dolar"] = df_dolar["Close"]

# Remover eventuais valores nulos decorrentes de desalinhamento de feriados
df.dropna(inplace=True)

Baixando dados históricos semanais...


/tmp/ipykernel_2357/3871360899.py:7: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_ativo = yf.download(ticker_ativo,start="2018-01-01",end="2026-11-10",interval="1wk",progress=False,)
/tmp/ipykernel_2357/3871360899.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_dolar = yf.download(ticker_dolar,start="2018-01-01",end="2026-11-10",interval="1wk",progress=False,)


In [13]:
# 2. Engenharia de Atributos (Features) Avançadas para o Semanal
# Retornos
df["Retorno_Semanal"] = df["Close"].pct_change()
df["Retorno_Dolar"] = df["Close_Dolar"].pct_change()

# Médias Móveis Estruturais
df["SMA_10w"] = df["Close"].rolling(window=10).mean()
df["SMA_40w"] = df["Close"].rolling(window=40).mean()  # Equivalente a ~200 dias
df["Dist_SMA40"] = (df["Close"] - df["SMA_40w"]) / df["SMA_40w"]


# Cálculo do MACD Semanal
def calcular_macd(
    data, slow=26, fast=12, signal=9
):  # Em barras semanais, 12/26 semanas
  exp1 = data.ewm(span=fast, adjust=False).mean()
  exp2 = data.ewm(span=slow, adjust=False).mean()
  macd = exp1 - exp2
  signal_line = macd.ewm(span=signal, adjust=False).mean()
  histogram = macd - signal_line
  return macd, signal_line, histogram


df["MACD"], df["MACD_Signal"], df["MACD_Hist"] = calcular_macd(df["Close"])


# Função RSI Semanal
def calcular_rsi(data, window=14):
  delta = data.diff()
  gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
  loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
  rs = gain / loss
  return 100 - (100 / (1 + rs))


df["RSI_14w"] = calcular_rsi(df["Close"], window=14)
df["Volatilidade_w"] = df["Retorno_Semanal"].rolling(window=10).std()

In [14]:
# 3. Definição do Alvo (Target Semanal)
# 1 se o fechamento da PRÓXIMA semana for maior que o desta semana, 0 caso contrário
df["Target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

# Limpar nulos gerados pelas janelas longas (ex: SMA de 40 semanas)
df.dropna(inplace=True)

# 4. Seleção de Features e Target
features = [
    "Retorno_Semanal",
    "Retorno_Dolar",
    "Dist_SMA40",
    "MACD_Hist",
    "RSI_14w",
    "Volatilidade_w",
    "Volume",
]
X = df[features]
y = df["Target"]

In [15]:
# 5. Divisão Temporal (Train/Test Split Cronológico)
train_size = int(len(df) * 0.8)
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

# 6. Treinamento da Random Forest (ajustada para evitar overfitting)
modelo = RandomForestClassifier(
    n_estimators=300,
    max_depth=4,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
modelo.fit(X_train, y_train)

RandomForestClassifier(max_depth=4, min_samples_leaf=5, n_estimators=300,
                       n_jobs=-1, random_state=42)

In [20]:
dot_data = export_graphviz(
    modelo.estimators_[0], # Seleciona a primeira árvore do conjunto
    out_file=None,
    feature_names=features,
    class_names=["Baixa", "Alta"],
    filled=True,
    rounded=True,
    special_characters=True,
)
graph = graphviz.Source(dot_data)
graph.render(f'arvore_decisao_{ticker_ativo}', format="png")
graph.view()

'arvore_decisao_EGIE3.SA.pdf'

In [18]:
# 7. Avaliação do Modelo no Teste
y_pred = modelo.predict(X_test)
print(f"\nAcurácia Semanal Avançada no Conjunto de Teste: {accuracy_score(y_test, y_pred):.4f}\n")
print("Relatório de Classificação:")
print(classification_report(y_test, y_pred))

# 8. Análise de Importância das Variáveis
importances = pd.Series(modelo.feature_importances_, index=features)
print("\nImportância dos Indicadores no Modelo Semanal:")
print(importances.sort_values(ascending=False))


Acurácia Semanal Avançada no Conjunto de Teste: 0.5176

Relatório de Classificação:
              precision    recall  f1-score   support

           0       0.47      0.17      0.25        40
           1       0.53      0.82      0.64        45

    accuracy                           0.52        85
   macro avg       0.50      0.50      0.45        85
weighted avg       0.50      0.52      0.46        85


Importância dos Indicadores no Modelo Semanal:
Volatilidade_w     0.212644
Volume             0.140684
RSI_14w            0.139085
Retorno_Semanal    0.134621
Retorno_Dolar      0.134438
Dist_SMA40         0.128361
MACD_Hist          0.110168
dtype: float64


In [19]:
# 1. Pegar a última linha de dados disponível no DataFrame (a semana mais recente fechada)
ultima_linha_features = X.iloc[[-1]]

# 2. Obter as probabilidades preditivas para esta última semana
prob_predita = modelo.predict_proba(ultima_linha_features)[0]

prob_queda = prob_predita[0] * 100
prob_alta = prob_predita[1] * 100

# 3. Data da última semana analisada
ultima_data = df.index[-1].strftime("%Y-%m-%d")

print(f"--- RELATÓRIO DE PREVISÃO PARA A PRÓXIMA SEMANA ({ticker_ativo}) ---")
print(f"Data de referência da última barra semanal: {ultima_data}")
print(f"Probabilidade estimada de QUEDA (0): {prob_queda:.2f}%")
print(f"Probabilidade estimada de ALTA (1):   {prob_alta:.2f}%")

# 4. Decisão baseada no Threshold (ex: 55% de confiança)
threshold_desejado = 55.0

if prob_alta >= threshold_desejado:
  print(
      "\n SINAL DO MODELO: **COMPRA / MANTER COMPRADO** (Confiança de alta"
      f" acima de {threshold_desejado}%)"
  )
elif prob_queda >= threshold_desejado:
  print(
      "\n SINAL DO MODELO: **FORA DO MERCADO / VENDA** (Confiança de baixa"
      f" acima de {threshold_desejado}%)"
  )
else:
  print(
      "\n SINAL DO MODELO: **NEUTRO / AGUARDAR** (O modelo está"
      " inconclusivo, abaixo do limiar de confiança)."
  )

--- RELATÓRIO DE PREVISÃO PARA A PRÓXIMA SEMANA (EGIE3.SA) ---
Data de referência da última barra semanal: 2026-11-09
Probabilidade estimada de QUEDA (0): 32.86%
Probabilidade estimada de ALTA (1):   67.14%

👉 SINAL DO MODELO: **COMPRA / MANTER COMPRADO** (Confiança de alta acima de 55.0%)
